# Week 9: Report Generator

SiteLens AI — Inspection report generation from structured damage records.

**Topic:** Prompt engineering, open-source LLMs  
**Dates:** May 17–21 2026  
**Deliverable:** Structured damage record → natural-language inspection report.

**Status:** Placeholder. To be implemented Week 9.

In [ ]:
import os
import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv()

assert os.environ.get("GEMINI_API_KEY"), "GEMINI_API_KEY not set"
print("API key present.")

In [ ]:
import pandas as pd
from src.translation.audience_translator import translate

df = pd.read_csv("../data/noto_crops/labels.csv")
print(f"Loaded {len(df):,} records. Columns: {list(df.columns)}")

In [ ]:
# Pick three records spanning the interesting cases
test_records = []

# (a) destroyed in fire zone — Asaichi market area
test_records.append(
    df[(df.damage_val == 1) & (df.gsi_fire == 1)].iloc[0].to_dict()
)
# (b) destroyed outside fire zone — seismic-only
test_records.append(
    df[(df.damage_val == 1) & (df.gsi_fire == 0)].iloc[0].to_dict()
)
# (c) survived
test_records.append(
    df[df.damage_val == 0].iloc[0].to_dict()
)

for i, rec in enumerate(test_records, 1):
    print(f"[{i}] s_fid={rec['s_fid']}  damage_val={rec['damage_val']}  "
          f"fire={rec['gsi_fire']}  tsunami={rec['gsi_tsunami']}  "
          f"slope={rec['gsi_slope_failure']}  conf={rec['conf']}")

## Insurance audience — Tuesday deliverable

Pass criterion: structured output, J-PIC category named, peril attribution follows fire > tsunami > slope_failure > seismic priority, no invented facts.

In [ ]:
for rec in test_records:
    print("\n" + "=" * 72)
    print(f"s_fid {rec['s_fid']} — damage_val {rec['damage_val']} "
          f"— fire {rec['gsi_fire']} — seismic-only {1 - rec['gsi_fire']}")
    print("=" * 72)

    result = translate(rec, audience="insurance")
    print("\n--- INSURANCE ADJUSTER ---")
    print(result["output_text"])

## Engineering + legal — drafted, evaluation Wednesday

Run these to confirm the prompts fire without errors. Output quality review is a Wednesday job.

In [ ]:
rec = test_records[0]  # fire-zone destroyed — most informative for all three lenses

for audience in ("engineering", "legal"):
    result = translate(rec, audience=audience)
    print(f"\n--- {audience.upper()} ---")
    print(result["output_text"])